Structured Output in LangChain

Structured Output in LangChain is a feature that forces an LLM to return its response in a predefined format (such as a Python dictionary, JSON object, or Pydantic model) instead of free-form text.

This makes the output consistent, predictable, and easy for applications to process.

Definition

Structured Output is the process of constraining an LLM's response to follow a specific schema or data model, ensuring that the returned data has the expected fields and data types.

In [4]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

Pydantic is a Python library used for data validation, parsing, and serialization based on Python type hints. It allows developers to define the structure of data using classes (called Pydantic models) and automatically validates that the data matches the expected types and rules.

In [1]:
from pydantic import BaseModel,Field

class Footballer(BaseModel):
    name:str = Field(description="name of the footballer/soccer player")
    jersey_number:int = Field(description="jersey number of the soccer player")
    club:str = Field(description="name of the club the soccer player/footballer plays for")

In [4]:
model_with_struct = model.with_structured_output(Footballer)
model_with_struct

_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-google-genai': '4.2.6'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash-Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x00000220C61A2560>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 're

In [8]:
### Un-Structured Output
model.invoke("greatest goal scorer of all time")

AIMessage(content='The title of the "greatest goal scorer of all time" is a hotly debated topic with no single, universally agreed-upon answer. However, when looking at **pure goal count across all senior professional matches**, the player most consistently cited and holding the record is:\n\n**Cristiano Ronaldo**\n\nAs of my last update, Cristiano Ronaldo has scored **over 890 senior professional goals**. This figure surpasses all other officially recognized goal scorers in the history of football.\n\n**Why the debate?**\n\nWhile Ronaldo\'s raw numbers are undeniable, other players are often brought into the discussion for various reasons:\n\n*   **Different Eras and Competition Levels:** Some argue that comparing players across different eras is difficult due to changes in the game, tactics, training, and the level of competition.\n*   **Unofficial Matches:** Some historical records include goals from unofficial matches or wartime leagues, which can inflate older players\' tallies.\n

In [9]:
### Structured Output
model_with_struct.invoke("greatest goal scorer of all time")

Footballer(name='Cristiano Ronaldo', jersey_number=7, club='Al Nassr')

Message Output Alongside Parsed Structure

In [12]:
from sqlalchemy import true
from pydantic import BaseModel,Field

class Footballer(BaseModel):
    name:str = Field(...,description="name of the footballer/soccer player")
    jersey_number:int = Field(...,description="jersey number of the soccer player")
    club:str = Field(...,description="name of the club the soccer player/footballer plays for")

model_with_struct = model.with_structured_output(Footballer,include_raw=True)
model_with_struct.invoke("greatest goal scorer of all time")

{'raw': AIMessage(content='{\n"name": "Cristiano Ronaldo",\n"jersey_number": 7,\n"club": "Al Nassr"\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f6b10-27ba-71e1-afd9-d89a3e64f7ef-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 29, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}}),
 'parsed': Footballer(name='Cristiano Ronaldo', jersey_number=7, club='Al Nassr'),
 'parsing_error': None}

Nested Structure of pydantic

In [7]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDict(BaseModel):
    title: str
    year: str
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None,description="Budget in millions")

model_w_struct = model.with_structured_output(MovieDict)

model_w_struct.invoke("give me details about spiderman no way home")


MovieDict(title='Spider-Man: No Way Home', year='2021', cast=[Actor(name='Tom Holland', role='Peter Parker / Spider-Man'), Actor(name='Zendaya', role='MJ'), Actor(name='Benedict Cumberbatch', role='Dr. Stephen Strange / Doctor Strange'), Actor(name='Jacob Batalon', role='Ned Leeds'), Actor(name='Jon Favreau', role='Happy Hogan'), Actor(name='Jamie Foxx', role='Max Dillon / Electro'), Actor(name='Willem Dafoe', role='Norman Osborn / Green Goblin'), Actor(name='Alfred Molina', role='Otto Octavius / Doctor Octopus'), Actor(name='Benedict Wong', role='Wong'), Actor(name='Tony Revolori', role='Eugene "Flash" Thompson'), Actor(name='Marisa Tomei', role='May Parker')], genres=['Action', 'Adventure', 'Fantasy', 'Sci-Fi'], budget=200.0)

TypedDict

TypedDict provides a simpler alternative using Python's built in typing, ideal when you dont need runtime validaiton.

In [8]:
from typing_extensions import TypedDict,Annotated

class MovieDetails(TypedDict):
    title: Annotated[str, ...,"title of the movie"]
    year: Annotated[int , ...,"the year when movie was released"]
    director: Annotated[str,...,"the director of the movie."]
    rating: Annotated[float,...,"the movie rating out of 10"]

model_with_typedict = model.with_structured_output(MovieDetails)
model_with_typedict.invoke("tell me about avengers movie")

{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.0}

In [9]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDict(TypedDict):
    title: str
    year: str
    cast: list[Actor]
    genres: list[str]
    budget: Annotated[float,...,"movie's budget in millions USD"]

model_w_struct = model.with_structured_output(MovieDict)

model_w_struct.invoke("give me details about spiderman no way home")


{'title': 'Spider-Man: No Way Home',
 'year': '2021',
 'cast': [{'name': 'Tom Holland', 'role': 'Peter Parker / Spider-Man'},
  {'name': 'Zendaya', 'role': 'MJ'},
  {'name': 'Benedict Cumberbatch',
   'role': 'Dr. Stephen Strange / Doctor Strange'},
  {'name': 'Jacob Batalon', 'role': 'Ned Leeds'},
  {'name': 'Jon Favreau', 'role': 'Happy Hogan'},
  {'name': 'Jamie Foxx', 'role': 'Max Dillon / Electro'},
  {'name': 'Willem Dafoe', 'role': 'Norman Osborn / Green Goblin'},
  {'name': 'Alfred Molina', 'role': 'Otto Octavius / Doctor Octopus'},
  {'name': 'Benedict Wong', 'role': 'Wong'},
  {'name': 'Marisa Tomei', 'role': 'May Parker'},
  {'name': 'Andrew Garfield', 'role': 'Peter Parker / Spider-Man'},
  {'name': 'Tobin Bell', 'role': 'J. Jonah Jameson'},
  {'name': 'Kirsten Dunst', 'role': 'Mary Jane Watson'},
  {'name': 'Charlie Cox', 'role': 'Matt Murdock'}],
 'genres': ['Action', 'Adventure', 'Fantasy', 'Science Fiction'],
 'budget': 200.0}

In [10]:
model.profile

{'name': 'Gemini 2.5 Flash-Lite',
 'release_date': '2025-06-17',
 'last_updated': '2025-06-17',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True}